In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import ast

print("Libraries imported successfully! Starting data pipeline... 🚀")

# Load Datasets 
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# Merge both dataframes on the 'title' column
movies = movies.merge(credits, on='title')

# Feature Selection
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Data Cleaning
movies.dropna(inplace=True)

# Helper function to parse JSON columns
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L 

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# Helper function for top 3 actors
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L 

movies['cast'] = movies['cast'].apply(convert3)

# Helper function for Director
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L 

movies['crew'] = movies['crew'].apply(fetch_director)

# Space Collapse Trick
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

# Feature Fusion
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Creating Final Structural DataFrame
new_df = movies[['movie_id', 'title', 'tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x)).apply(lambda x: x.lower())

print("Data preprocessing complete. TF-IDF Vectorization starting... 🧠")


tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vector = tfidf.fit_transform(new_df['tags']).toarray()

# Mathematical Inference Matrix using Cosine Similarity
similarity = cosine_similarity(vector)

# Model Serialization using Pickle
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("TF-IDF Model trained and .pkl files generated successfully inside ml-core! 🔥🚀")

Libraries imported successfully! Starting data pipeline... 🚀


C:\Users\Arpit Carpenter\AppData\Local\Temp\ipykernel_2504\1046608139.py:76: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x)).apply(lambda x: x.lower())


Data preprocessing complete. TF-IDF Vectorization starting... 🧠
TF-IDF Model trained and .pkl files generated successfully inside ml-core! 🔥🚀
